# flash attention, tier 2 — the real kernel (Triton)

companion notebook to *flash attention, from first principles*
(https://adithyag73.github.io/first_principles/flash-attention/).

tier 1 (numpy) proved the mathematics exact. this notebook proves the **speed**:
a Triton kernel where the tiles genuinely live in SRAM, benchmarked against
naive materialise-S-and-P attention on a real GPU. run every cell (Kaggle: GPU T4/P100
or better via Settings → Accelerator) and u get ur own version of the paper's Fig. 2.

fun fact: Triton's creator, Phil Tillet, is also the person who first flipped the
flash attention loop order — the flip re-derived in section 6 of the article.

In [ ]:
import torch, triton, triton.language as tl
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(0))

## the kernel — Algorithm 1 with the FA-2 flip
askers (Q blocks) take residence — one per program instance, embarrassingly parallel —
and the shelf (K, V blocks) parades past. notebooks `m, r` live in registers;
the running blend `acc` never leaves SRAM until its final, single write. every line
maps to a sentence in the article: the local queen, the repair factor, the two-stroke update.

In [ ]:
@triton.jit
def flash_fwd(Q, K, V, O, N, scale,
              stride_qm, stride_qd, stride_km, stride_kd,
              stride_vm, stride_vd, stride_om, stride_od,
              BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, D: tl.constexpr):
    pid = tl.program_id(0)                      # which asker block am i?
    offs_m = pid * BLOCK_M + tl.arange(0, BLOCK_M)   # my Q rows
    offs_d = tl.arange(0, D)
    # residence: my Q block, loaded ONCE (the flip: askers sit, shelf commutes)
    q = tl.load(Q + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd,
                mask=offs_m[:, None] < N, other=0.0)
    # notebooks — sentinels: queen -inf, room total 0, blend 0 (wet paint stays here)
    m_i = tl.full((BLOCK_M,), float('-inf'), tl.float32)
    r_i = tl.zeros((BLOCK_M,), tl.float32)
    acc = tl.zeros((BLOCK_M, D), tl.float32)
    # the shelf parades
    for j0 in range(0, N, BLOCK_N):
        offs_n = j0 + tl.arange(0, BLOCK_N)
        k = tl.load(K + offs_n[:, None] * stride_km + offs_d[None, :] * stride_kd,
                    mask=offs_n[:, None] < N, other=0.0)
        v = tl.load(V + offs_n[:, None] * stride_vm + offs_d[None, :] * stride_vd,
                    mask=offs_n[:, None] < N, other=0.0)
        s = tl.dot(q, tl.trans(k)) * scale                 # mint the tile (SRAM only)
        s = tl.where(offs_n[None, :] < N, s, float('-inf'))
        m_tilde = tl.max(s, 1)                             # local queens, per row
        m_new = tl.maximum(m_i, m_tilde)                   # crown contest
        conv = tl.exp(m_i - m_new)                         # repair factor e^(m_old - m_new)
        p = tl.exp(s - m_new[:, None])                     # newcomers in new currency
        r_new = conv * r_i + tl.sum(p, 1)                  # line 11: repair, then admit
        acc = acc * conv[:, None] + tl.dot(p.to(v.dtype), v)  # line 12 (unnormalised)
        m_i, r_i = m_new, r_new
    acc = acc / r_i[:, None]                               # one final divide per row
    tl.store(O + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od,
             acc, mask=offs_m[:, None] < N)                # O leaves ONCE, finished

def flash_attention(Q, K, V, BLOCK_M=64, BLOCK_N=64):
    N, d = Q.shape
    O = torch.empty_like(Q, dtype=torch.float32)
    grid = (triton.cdiv(N, BLOCK_M),)
    flash_fwd[grid](Q, K, V, O, N, 1.0,
                    Q.stride(0), Q.stride(1), K.stride(0), K.stride(1),
                    V.stride(0), V.stride(1), O.stride(0), O.stride(1),
                    BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, D=d)
    return O

## exactness first — always

In [ ]:
def naive_attention(Q, K, V):
    S = Q @ K.T                       # N x N materialised — the crime
    P = torch.softmax(S, dim=-1)      # N x N materialised — the accomplice
    return P @ V

torch.manual_seed(0)
N, d = 1024, 64
Q = torch.randn(N, d, device='cuda'); K = torch.randn(N, d, device='cuda'); V = torch.randn(N, d, device='cuda')
ref, out = naive_attention(Q, K, V), flash_attention(Q, K, V)
print('max abs diff:', (ref - out).abs().max().item())
assert torch.allclose(ref, out, atol=1e-4), 'exactness failed'
print('exactness: PASS')

## the race — ur own Fig. 2
these numbers feed widget 6 of the article. paste them there when done.

In [ ]:
import time
def bench(fn, *args, iters=50):
    for _ in range(5): fn(*args)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(iters): fn(*args)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3

print(f'{"N":>6} {"naive ms":>10} {"flash ms":>10} {"speedup":>9}')
results = []
for N in [256, 512, 1024, 2048, 4096, 8192]:
    Q = torch.randn(N, d, device='cuda'); K = torch.randn(N, d, device='cuda'); V = torch.randn(N, d, device='cuda')
    tn, tf = bench(naive_attention, Q, K, V), bench(flash_attention, Q, K, V)
    results.append((N, tn, tf))
    print(f'{N:>6} {tn:>10.3f} {tf:>10.3f} {tn/tf:>8.1f}x')
print()
print('paste into widget6-benchmark.html:')
print('const DATA =', [[n, round(a,3), round(b,3)] for n,a,b in results], ';')

## what to expect
the gap should WIDEN with N — the naive version's N² ghost traffic grows quadratically
while the kernel's traffic grows like N²d²/M (section 7's theorem). at large N the naive
version may simply run out of memory trying to materialise S — which is the article's
entire argument, enforced by the hardware itself.